# [0.6] How to Know When an Interpretability Result Is Fake - Exercises

## Core Question

How do you tell whether a beautiful interpretability result is evidence, or only a story that survived no controls?

## Learning Objectives

- Detect direct label leakage with a shifted no-leak control.
- Detect cherry-picked examples with population statistics.
- Detect probe memorization with held-out controls.
- Reject weak steering directions with random-direction baselines.
- Aggregate the diagnostics into a CUDA-backed signature report.

### Exercise - fake-result diagnostics ladder

> Difficulty: medium  
> Importance: high

This is a GT-0 skepticism lab. Every impressive-looking result below is deliberately fake,
so the right outcome is a negative result: each detector should reject the bogus claim.


<details>
<summary>Help - what to watch for</summary>

A detector that only works on one hardcoded fixture is itself suspicious. Each report
function in this notebook has a default toy oracle, but the tests also pass alternate
tensors so your implementation has to compute the metric from inputs.

</details>


In [ ]:
import sys
from dataclasses import dataclass
from pathlib import Path

import torch as t

chapter = "chapter0_fundamentals"
section = "part6_fake_interpretability_results"
root_dir = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section

if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))
if str(exercises_dir) not in sys.path:
    sys.path.append(str(exercises_dir))

import part6_fake_interpretability_results.tests as tests

GT_TIER = "GT-0"
EXERCISE_ID = "0_6_how_to_know_when_an_interpretability_result_is_fake"
EXPECTED_RUNTIME = "20-30 minutes for exercises; seconds for the CUDA diagnostic preflight"
REQUIRES_GPU = True


In [ ]:
@dataclass(frozen=True)
class LabelLeakageReport:
    leaked_feature_index: int
    leaked_feature_accuracy: float
    shifted_no_leak_accuracy: float
    accuracy_gap: float
    detects_leakage: bool


@dataclass(frozen=True)
class CherryPickReport:
    selected_mean_effect: float
    population_mean_effect: float
    population_median_effect: float
    inflation_ratio: float
    detects_cherry_picking: bool


@dataclass(frozen=True)
class ProbeOverfitReport:
    train_accuracy: float
    heldout_accuracy: float
    generalization_gap: float
    detects_overfit: bool


@dataclass(frozen=True)
class FakeRandomDirectionControlReport:
    claimed_effect: float
    random_p95_effect: float
    effect_gap: float
    passes_random_control: bool
    detects_random_direction_failure: bool


@dataclass(frozen=True)
class FakeResultAuditReport:
    leakage_detected: bool
    cherry_pick_detected: bool
    probe_overfit_detected: bool
    random_direction_failure_detected: bool
    all_bogus_results_flagged: bool


## Label Leakage

Implement threshold-at-zero binary accuracy, then detect a feature column that directly encodes the label. The shifted/no-leak control must fail; perfect accuracy alone is not evidence.

<details>
<summary>Expected output</summary>

```text
All tests in `test_binary_accuracy_thresholds_signed_scores` passed!
All tests in `test_label_leakage_report_flags_direct_label_feature` passed!
All tests in `test_label_leakage_report_uses_supplied_feature_indices` passed!
```

</details>

In [ ]:
def binary_accuracy(scores: t.Tensor, labels: t.Tensor) -> float:
    # EXERCISE
    # YOUR CODE HERE
    raise NotImplementedError()
    # END EXERCISE


def label_leakage_report(
    features: t.Tensor | None = None,
    labels: t.Tensor | None = None,
    *,
    leaked_feature_index: int = 0,
    shifted_feature_index: int = 1,
    min_gap: float = 0.5,
) -> LabelLeakageReport:
    # EXERCISE
    # YOUR CODE HERE
    raise NotImplementedError()
    # END EXERCISE


tests.test_binary_accuracy_thresholds_signed_scores(binary_accuracy)
tests.test_label_leakage_report_flags_direct_label_feature(label_leakage_report)
tests.test_label_leakage_report_uses_supplied_feature_indices(label_leakage_report)


<details>
<summary>Solution</summary>

```python
def binary_accuracy(scores, labels):
    labels = labels.long().flatten()
    predictions = scores.flatten().gt(0).long()
    return float(predictions.eq(labels).double().mean().item())


def label_leakage_report(features=None, labels=None, *, leaked_feature_index=0, shifted_feature_index=1, min_gap=0.5):
    if features is None or labels is None:
        labels = t.tensor([0, 1, 0, 1, 0, 1], dtype=t.long)
        signed = labels.float() * 2 - 1
        features = t.stack([signed, -signed], dim=1)
    leaked_accuracy = binary_accuracy(features[:, leaked_feature_index], labels)
    shifted_accuracy = binary_accuracy(features[:, shifted_feature_index], labels)
    gap = leaked_accuracy - shifted_accuracy
    return LabelLeakageReport(leaked_feature_index, leaked_accuracy, shifted_accuracy, gap, leaked_accuracy >= 0.95 and gap >= min_gap)
```

</details>

## Cherry-Picked Evidence

Selected examples must be compared to the full population. A representative selection should not be rejected.

<details>
<summary>Expected output</summary>

```text
All tests in `test_cherry_pick_report_compares_selected_to_population` passed!
All tests in `test_cherry_pick_report_rejects_representative_selection` passed!
```

</details>

In [ ]:
def cherry_pick_report(
    effects: t.Tensor | None = None,
    selected_indices: t.Tensor | list[int] | None = None,
    *,
    min_inflation: float = 3.0,
    median_multiplier: float = 5.0,
) -> CherryPickReport:
    # EXERCISE
    # YOUR CODE HERE
    raise NotImplementedError()
    # END EXERCISE


tests.test_cherry_pick_report_compares_selected_to_population(cherry_pick_report)
tests.test_cherry_pick_report_rejects_representative_selection(cherry_pick_report)


<details>
<summary>Solution</summary>

```python
if effects is None:
    effects = t.tensor([0.02, 0.03, ..., 1.40, 1.55, 1.70])
selected = effects[-3:] if selected_indices is None else effects[selected_indices]
selected_mean = float(selected.mean().item())
population_mean = float(effects.mean().item())
population_median = float(effects.median().item())
inflation = selected_mean / population_mean
```

Return a `CherryPickReport` and require both high inflation and a large median-relative selected mean.

</details>

## Probe Overfitting

A memorizing probe is not evidence unless it survives held-out examples. The positive-control test passes a probe that generalizes.

<details>
<summary>Expected output</summary>

```text
All tests in `test_probe_overfit_report_requires_heldout_gap` passed!
All tests in `test_probe_overfit_report_rejects_generalizing_probe` passed!
```

</details>

In [ ]:
def probe_overfit_report(
    train_predictions: t.Tensor | None = None,
    train_labels: t.Tensor | None = None,
    heldout_predictions: t.Tensor | None = None,
    heldout_labels: t.Tensor | None = None,
    *,
    min_train_accuracy: float = 0.95,
    max_heldout_accuracy: float = 0.6,
    min_gap: float = 0.35,
) -> ProbeOverfitReport:
    # EXERCISE
    # YOUR CODE HERE
    raise NotImplementedError()
    # END EXERCISE


tests.test_probe_overfit_report_requires_heldout_gap(probe_overfit_report)
tests.test_probe_overfit_report_rejects_generalizing_probe(probe_overfit_report)


<details>
<summary>Solution</summary>

Compute train and held-out accuracy separately, then flag overfit only when train accuracy is high, held-out accuracy is weak, and the gap is large.

</details>

## Random-Direction Controls

A steering direction should beat a random-direction control distribution by a real margin. The detector should reject weak claims and accept a strong positive control.

<details>
<summary>Expected output</summary>

```text
All tests in `test_random_direction_control_report_rejects_weak_claim` passed!
All tests in `test_random_direction_control_report_accepts_strong_claim` passed!
```

</details>

In [ ]:
def random_direction_control_report(
    behavior_direction: t.Tensor | None = None,
    claimed_direction: t.Tensor | None = None,
    random_directions: t.Tensor | None = None,
    *,
    required_margin: float = 0.25,
) -> FakeRandomDirectionControlReport:
    # EXERCISE
    # YOUR CODE HERE
    raise NotImplementedError()
    # END EXERCISE


tests.test_random_direction_control_report_rejects_weak_claim(random_direction_control_report)
tests.test_random_direction_control_report_accepts_strong_claim(random_direction_control_report)


<details>
<summary>Solution</summary>

Normalize the behavior, claimed, and random directions. Compare the claimed absolute dot product to the random-control p95. The claim passes only when `claimed_effect - random_p95 >= required_margin`.

</details>

## Aggregate Audit

The section-level audit should pass only when every known bogus result is flagged.

<details>
<summary>Expected output</summary>

```text
All tests in `test_fake_result_audit_report_aggregates_all_failure_modes` passed!
All tests in `test_notebook_contract` passed!
```

</details>

In [ ]:
def fake_result_audit_report(
    leakage: LabelLeakageReport | None = None,
    cherry_pick: CherryPickReport | None = None,
    overfit: ProbeOverfitReport | None = None,
    random_direction: FakeRandomDirectionControlReport | None = None,
) -> FakeResultAuditReport:
    # EXERCISE
    # YOUR CODE HERE
    raise NotImplementedError()
    # END EXERCISE


def leakage_diagnostic() -> dict:
    return label_leakage_report().__dict__


def cherry_pick_diagnostic() -> dict:
    return cherry_pick_report().__dict__


def probe_overfit_diagnostic() -> dict:
    return probe_overfit_report().__dict__


def random_direction_diagnostic() -> dict:
    return random_direction_control_report().__dict__


def run_smoke_test(cpu: bool = True) -> dict:
    _ = cpu
    audit = fake_result_audit_report().__dict__
    return {
        "leakage": leakage_diagnostic(),
        "cherry_pick": cherry_pick_diagnostic(),
        "probe_overfit": probe_overfit_diagnostic(),
        "random_direction": random_direction_diagnostic(),
        "audit": audit,
        "contract_passed": audit["all_bogus_results_flagged"],
    }


tests.test_fake_result_audit_report_aggregates_all_failure_modes(fake_result_audit_report)
tests.test_notebook_contract(run_smoke_test)


<details>
<summary>Solution</summary>

Call each detector, store the four booleans, and return `all_bogus_results_flagged = all(flags)`. The point of the aggregate report is to make missing controls visible.

</details>

## Signature Result

![Fake interpretability signature result](expected_outputs/fake_interpretability_signature_result.svg)

The committed CUDA report should show that all four injected bogus claims were rejected.

<details>
<summary>Expected output</summary>

```text
All tests in `test_committed_gpu_report_records_fake_result_signature` passed!
preflight_passed: true
all_bogus_results_flagged: true
peak_vram_gb: < 1.0
```

</details>

In [ ]:
def _load_committed_gpu_report() -> dict:
    import json

    report = json.loads((section_dir / "verification_report.json").read_text())
    assert report["accepted"] and report["tests_passed"]
    gpu = report["metrics"]["gpu_test"]
    assert gpu["cuda_available"]
    assert gpu["within_vram_budget"]
    return gpu


def run_gpu_test(max_vram_gb: float = 24.0) -> dict:
    gpu = _load_committed_gpu_report()
    assert gpu["peak_vram_gb"] <= max_vram_gb
    return gpu


def run_full_experiment(max_vram_gb: float = 24.0) -> dict:
    return run_gpu_test(max_vram_gb=max_vram_gb)


tests.test_committed_gpu_report_records_fake_result_signature()

gpu = _load_committed_gpu_report()
{key: gpu[key] for key in [
    "device",
    "preflight_passed",
    "all_bogus_results_flagged",
    "peak_vram_gb",
] if key in gpu}


## Limitations

This notebook proves that the detectors catch known synthetic failures. It does not prove a real model mechanism, and it does not cover every possible source of misleading interpretability evidence.

## Bonus - Anomaly Hunting

Try weakening each failure mode: 80% leakage instead of 100%, a less extreme selected subset, a held-out probe at 70%, or a claimed direction that barely beats random p95. Decide whether the result should fail, warn, or pass.